# A1 cross-subject re-ID — EEGNet (with volts→microvolts fix)

Runs experiment `02_closed_set_reid` for all three victim families on the full 104-subject PhysioNet motor-imagery split. Designed to take ~25–30 minutes on a Colab L4. Total session disk use: ~3 GB (1.7 GB EDFs + 2.3 GB cached windows).

After the run finishes, send back the two small JSON files from Cell 7 (`results/02_closed_set_reid.json` and `runs/<run_id>/meta.json`). Those get committed to the canonical locations in the repo. The figure regenerates locally from the result JSON.

**Runtime → L4 GPU.** Set via Runtime → Change runtime type → L4 GPU.

**One thing — don't `Save a copy in GitHub` from Colab.** Just run the cells and use the download cell at the end. Saving back overwrites this source file with a Colab-mangled version (cell IDs change, output cells get committed, diff becomes 200+ lines of noise). The result JSON + run metadata are everything I need.

## 1. Clone the repo (always pulls latest `main`)

In [1]:
!rm -rf /content/bci-identity-leakage
!git clone https://github.com/manrajmondair/bci-identity-leakage.git /content/bci-identity-leakage
%cd /content/bci-identity-leakage
!git rev-parse HEAD

Cloning into '/content/bci-identity-leakage'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 75 (delta 26), reused 59 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (75/75), 66.43 KiB | 9.49 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/bci-identity-leakage
3733900c7f6e761561364c29fd102fcafc6af229


## 2. Confirm GPU is attached

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

name, memory.total [MiB]
NVIDIA L4, 23034 MiB
torch: 2.10.0+cu128 | cuda: True | device: NVIDIA L4


## 3. Install project deps

Colab already ships with torch / numpy / scipy / pandas / scikit-learn / matplotlib. We only install the EEG-specific packages and force the matching torchaudio (Colab's preinstalled torchaudio is sometimes a different ABI build).

In [3]:
import torch
tv = torch.__version__.split('+')[0]   # e.g. '2.5.1'
!pip install --quiet "torchaudio=={tv}"
!pip install --quiet mne moabb pyriemann braindecode opacus httpx tqdm rich

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 161.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.7/837.7 kB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.8/138.8 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 497.9/497.9 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.9/308.9 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.2/254.2 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.4/178.4 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 160.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.5/268.5 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 162.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver doe

## 4. Prefetch PhysioNet imagery (~2–3 min on archive mirror)

The archive mirror (`archive.physionet.org`) serves the EDFs ~10× faster than the versioned URL on `physionet.org`. Total download: 1.35 GB × 11 MB/s ≈ 2 minutes.

In [ ]:
!cd /content/bci-identity-leakage && python -m data.prefetch_direct --runs imagery --workers 8

## 5. Run A1 cross-subject re-ID — all three victim families

EEGNet now uses `input_scale=1e6` to convert volts → microvolts before the network's first conv. The previous chance-level EEGNet result was caused by gradients vanishing at the volt scale; this run replaces that EEGNet row of the A1 table.

Expected wall: ~25 min (EEGNet train ~8 min on L4, FBCSP ~7 min, Riemann ~1 min, attacks ~5 min).

In [ ]:
!cd /content/bci-identity-leakage && PYTHONUNBUFFERED=1 python -u -m experiments.02_closed_set_reid --all

## 6. Emit run metadata + result JSON

Writes the canonical result JSON (already populated by Cell 5) and a `runs/<run_id>/meta.json` capturing git SHA, hardware, wall time, args, timestamps. Send me both files (or just paste the printed blocks) — I'll commit them.

In [ ]:
# Robust paste-back: survives a Colab reconnect and prints the result to copy.
import json, os, subprocess, datetime, platform
import torch
REPO = '/content/bci-identity-leakage'
if not os.path.isdir(REPO):
    raise SystemExit('Runtime was reset and the clone is gone. '
                     'Use Runtime > Run all to redo this notebook end to end.')
os.chdir(REPO)
EXPERIMENT, ARGS, TAG = 'experiments.02_closed_set_reid', ['--all'], 'a1_cross'
RESULTS, FIGURE = ['results/02_closed_set_reid.json'], ['figures/02_closed_set_reid.pdf']
try:
    git_sha = subprocess.check_output(['git', '-C', REPO, 'rev-parse', 'HEAD']).decode().strip()
except Exception as exc:
    print(f'  git unavailable ({exc}); using "unknown"'); git_sha = 'unknown'
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
now_utc = datetime.datetime.now(datetime.timezone.utc).replace(microsecond=0).isoformat().replace('+00:00', 'Z')
run_id = now_utc.replace(':', '').replace('-', '').rstrip('Z') + f'_{TAG}_{git_sha[:7]}'
meta = {'run_id': run_id, 'experiment': EXPERIMENT, 'args': ARGS, 'git_sha': git_sha,
        'hardware': f'Colab {gpu}', 'platform': platform.platform(),
        'torch_version': torch.__version__, 'completed_at_utc': now_utc,
        'outputs': RESULTS + FIGURE}
os.makedirs(f'runs/{run_id}', exist_ok=True)
with open(f'runs/{run_id}/meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('=== run metadata ==='); print(json.dumps(meta, indent=2)); print()
missing = [r for r in RESULTS if not os.path.exists(r)]
if missing:
    raise SystemExit(f'!!! missing {missing} — the experiment cell did not finish; re-run it then this cell.')
for r in RESULTS:
    print(f'=== {r} ==='); print(open(r).read()); print()


## 7. Download artifacts (drag both into the chat)

Two small files. The figure stays in Colab — I regenerate it locally from the result JSON.

In [ ]:
# (paste-back prints the result above; no download needed)
